# Manual baseline runs

Use the project's `.venv` Python kernel. This notebook reads arguments directly from `main.py`, parses the selected baseline configuration, and calls its training function in this kernel for debugging.

Change run arguments in `main.py`, then rerun the configuration cells. No small-run, prompt, seed, or output-directory overrides are applied here. Runs use the same output paths and checkpoint/resume behavior as the launcher.

Use a Slurm GPU allocation for training; submit experiments through `sbatch run_experiments.sh` per project policy. Configuration inspection is separate from training.

Restart the kernel after changing environment settings, then run the setup cell first. It reads simple `export NAME=value` assignments from `run_experiments.sh` and loads `.env` before importing the launcher or model libraries. It also configures a valid CA bundle for HTTPS; shell commands in the batch script are not executed.


In [ ]:
from pathlib import Path
import os
import sys

# Works when opened from the repository root or clover/exp.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "main.py").is_file() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the Clover repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Apply the batch script's simple export assignments in order, without
# executing its activation, directory changes, srun, or job submission.
from io import StringIO
from dotenv import dotenv_values, load_dotenv
import ssl
import tempfile

for raw_line in (ROOT / "run_experiments.sh").read_text().splitlines():
    line = raw_line.strip()
    if line.startswith("export ") and "=" in line:
        assignments = dotenv_values(stream=StringIO(line))
        for name, value in assignments.items():
            if value is not None:
                os.environ[name] = value
    elif line in ("source .env", ". .env"):
        load_dotenv(ROOT / ".env", override=True)
# Also load .env if the batch script no longer explicitly sources it.
load_dotenv(ROOT / ".env", override=True)

# This cluster's Python may reference a nonexistent OpenSSL CA location.
# Preserve configured certificate settings; otherwise use the system bundle.
if not os.environ.get("SSL_CERT_FILE"):
    ca_bundle = os.environ.get("REQUESTS_CA_BUNDLE") or os.environ.get("CURL_CA_BUNDLE")
    if not ca_bundle:
        ca_bundle = ssl.get_default_verify_paths().cafile
    if not ca_bundle:
        ca_bundle = "/etc/ssl/certs/ca-certificates.crt"
    if not Path(ca_bundle).is_file():
        raise FileNotFoundError("Configure SSL_CERT_FILE with a readable CA bundle.")
    os.environ["SSL_CERT_FILE"] = ca_bundle
# Requests and urllib/OpenSSL use different environment variable names.
os.environ.setdefault("REQUESTS_CA_BUNDLE", os.environ["SSL_CERT_FILE"])
ssl.create_default_context()  # Check the trust configuration; never disable TLS verification.
tempfile.tempdir = None  # Honor TMPDIR even if Jupyter previously cached a temp path.

print("Repository:", ROOT)
print("Python:", sys.executable)
print("Slurm job:", os.environ.get("SLURM_JOB_ID", "not allocated"))
print("Batch environment and .env loaded; HTTPS certificate verification enabled.")


Repository: /aiau010_scratch/azm0269/clover
Python: /aiau010_scratch/azm0269/clover/.venv/bin/python
Slurm job: 25372
Batch environment and .env loaded; HTTPS certificate verification enabled.


## Load arguments from main.py

Select the first enabled baseline and first experimental seed from the launcher. Reloading picks up edits to `main.py` without restarting the kernel. `build_default_argv` supplies the training arguments, including GPU IDs and output directory; unspecified fields retain the baseline defaults.


In [2]:
import importlib
import main as launcher

launcher = importlib.reload(launcher)

CONFIG_TYPES = {
    # "b2diffurl": "B2DiffuRLConfig", "ddpo": "DDPOConfig", "dpok": "DPOKConfig",
    # "md3po": "MD3POConfig", "md3po_sac": "MD3POSACConfig", "emo": "EMOConfig",
    # "emo_v2": "EMOV2V2Config", 
    "emo_v3": "EMOV3Config", 
    # "sqdf": "SQDFConfig",
}
BASELINE, module_name = launcher.BASELINES[0]
SEED = launcher.EXPERIMENT_SEEDS[0]
print(BASELINE, SEED)


emo_v3 123


In [ ]:
from dataclasses import asdict
from pprint import pprint
from clover.baselines.common import parse_config

baseline = importlib.import_module(module_name)
config_type = getattr(baseline, CONFIG_TYPES[BASELINE])
argv = launcher.build_default_argv(module_name, seed=SEED)
config = parse_config(config_type, baseline.__doc__ or BASELINE, argv=argv[1:])
pprint(asdict(config))


## Run and debug

Training calls the selected module's `train(config)` directly, retaining the baseline's reward, evaluation, checkpoint, and trajectory logic. Imports may save the standard prompt split, just as the baseline CLI does.

If training raises an exception, run `%debug` in a new cell to inspect stack frames and local tensors (`up`, `down`, `p variable`, `q`). Use source breakpoints or `%pdb on` before training for interactive debugging. Running this cell again may resume the checkpoint at the output path defined by `main.py`.


In [4]:
import time
import torch

if not os.environ.get("SLURM_JOB_ID"):
    raise RuntimeError("Run training in a Slurm GPU allocation; use sbatch run_experiments.sh for experiments.")
if not torch.cuda.is_available():
    raise RuntimeError("The notebook kernel has no CUDA GPU available.")
if not 1 <= len(config.gpu_ids) <= 2:
    raise ValueError("Select one or two allocated GPUs at most.")
if any(i < 0 or i >= torch.cuda.device_count() for i in config.gpu_ids):
    raise ValueError("config.gpu_ids must refer to visible logical CUDA devices.")
print("GPU:", torch.cuda.get_device_name(config.gpu_ids[0]))
started = time.perf_counter()
history = baseline.train(config)
elapsed_seconds = time.perf_counter() - started
print(f"Finished in {elapsed_seconds:.2f}s")


GPU: NVIDIA H200
cuda:0 torch.float16 gpu_ids=[0]


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Trainable LoRA parameters kept in FP32 for optimizer stability
xFormers memory efficient attention enabled
Training 3,200,260 LoRA parameters


  0%|          | 0/50 [00:00<?, ?it/s]

...

{'loss': -0.24980577309805976, 'policy_loss': 0.15820602012551105, 'entropy': 0.09159975302386648, 'entropy_bonus': 0.09159975302386648, 'beta': 30.0, 'reward_scale': 30.0, 'importance_ratio': 0.9999923080814128, 'reward_mean': 0.21264567971229553, 'reward_std': 0.07326129078865051, 'soft_q_mean': 0.867847204208374, 'grad_norm_mean': 0.007491784915328026, 'skipped_updates': 0, 'selected_samples': 303, 'current_reward_mean': 0.12481340765953064, 'current_reward_std': 0.024516059085726738, 'replay_reward_mean': 0.13083554804325104, 'replay_reward_std': 0.017789386212825775, 'replay_samples': 47, 'replay_source_epoch': 10, 'epoch': 11, 'seed': 123, 'learning_rate': 9.140576474687264e-05}
{'loss': -0.25763896231976696, 'policy_loss': 0.16603216484797245, 'entropy': 0.0916067974375827, 'entropy_bonus': 0.0916067974375827, 'beta': 30.0, 'reward_scale': 30.0, 'importance_ratio': 0.9999922230535624, 'reward_mean': 0.21346436440944672, 'reward_std': 0.07258515805006027, 'soft_q_mean': 0.8827065

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

[transformers] DebertaModel LOAD REPORT from: microsoft/deberta-large-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
config              | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 
classifier.bias     | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Error during conversion: ReadTimeout('The read operation timed out')


KeyboardInterrupt: 

## Inspect saved results

`history` contains returned training metrics. Evaluation files are under `config.output_dir/evals`, using the output directory supplied by `main.py`. Training/replay artifacts retain the baseline's existing paths and formats under `clover/data`.


In [ ]:
import json
from clover.utils.baseline_utils import save_json

pprint(history)
run_summary = {
    "baseline": BASELINE,
    "seed": config.seed,
    "output_dir": config.output_dir,
    "execution_time": elapsed_seconds,
    "gpu_ids": config.gpu_ids,
}
save_json(Path(config.output_dir) / "manual_execution_time.json", run_summary)
eval_dir = Path(config.output_dir) / "evals"
for path in sorted(eval_dir.glob("*.json")):
    print(path.relative_to(ROOT))
    payload = json.loads(path.read_text())
    pprint(payload[-2:] if isinstance(payload, list) else payload)
